# 부록 02. init_chat_model로 모델 초기화하기

LangChain 1.0에서는 `init_chat_model` 함수를 통해 다양한 LLM 제공자의 모델을 통일된 방식으로 초기화할 수 있습니다.

## 학습 목표
- `init_chat_model` 함수 사용법 이해
- 다양한 모델 제공자(OpenAI, Anthropic, Google, Azure 등) 설정 방법
- 모델 파라미터 조정 방법 학습
- 주요 호출 방식(invoke, stream, batch) 활용
- 멀티모달 입력 및 추론(reasoning) 기능 활용

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("환경 변수 상태:")
print(f"- OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")
print(f"- ANTHROPIC_API_KEY: {'설정됨' if os.getenv('ANTHROPIC_API_KEY') else '미설정'}")

## 2. init_chat_model 기본 사용법

`init_chat_model`은 모델 식별자 문자열을 받아 해당 모델을 초기화합니다.

### 모델 식별자 형식
- `"model_name"` 형식으로 모델만 지정 (예: `"gpt-4o-mini"`, `"claude-sonnet-4-5-20250929"`)
- `"provider:model_name"` 형식으로 명시적 지정 (예: `"openai:gpt-4o"`, `"anthropic:claude-sonnet-4-5-20250929"`)

### 지원하는 주요 모델 제공자
- **OpenAI**: `gpt-4o`, `gpt-4o-mini`, `o1`, `o1-mini`, `o3-mini` 등
- **Anthropic**: `claude-sonnet-4-5-20250929`, `claude-opus-4-6` 등
- **Google**: `gemini-2.5-flash-lite`, `gemini-2.0-flash-exp` 등
- **Azure OpenAI**: `azure_openai:gpt-4o`
- **AWS Bedrock**: `bedrock_converse:anthropic.claude-3-5-sonnet-20240620-v1:0`

In [ ]:
from langchain.chat_models import init_chat_model

# 기본 사용법: 모델명으로 초기화
model = init_chat_model("gpt-4o-mini")

# 간단한 호출
response = model.invoke("안녕하세요! 반갑습니다.")
print(f"모델 응답: {response.content}")

In [ ]:
# provider:model_name 형식 사용
model_explicit = init_chat_model("openai:gpt-4o-mini")

response = model_explicit.invoke("LangChain 1.0에 대해 한 문장으로 설명해주세요.")
print(f"응답: {response.content}")

## 3. 모델 파라미터 설정

`init_chat_model`은 다양한 파라미터를 지원합니다:

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `temperature` | 응답의 창의성 조절 (0~2) | 1.0 |
| `max_tokens` | 최대 출력 토큰 수 | 모델 기본값 |
| `timeout` | 요청 타임아웃 (초) | None |
| `max_retries` | 재시도 횟수 | 2 |

In [ ]:
# 파라미터 설정 예제
model_configured = init_chat_model(
    "gpt-4o-mini",
    temperature=0,       # 결정적 응답 (창의성 낮춤)
    max_tokens=500,      # 최대 500 토큰
    timeout=30,          # 30초 타임아웃
)

# 같은 질문을 여러 번 해도 일관된 답변
for i in range(2):
    response = model_configured.invoke("1+1은?")
    print(f"시도 {i+1}: {response.content}")

## 4. 다양한 모델 제공자 사용하기

In [ ]:
# OpenAI 모델들
models_to_test = [
    "gpt-4o-mini",        # GPT-4o Mini (빠르고 저렴)
    "gpt-4o",             # GPT-4o (고성능)
]

question = "AI 에이전트를 한 문장으로 설명해주세요."

for model_name in models_to_test:
    try:
        model = init_chat_model(model_name, temperature=0)
        response = model.invoke(question)
        print(f"\n[{model_name}]")
        print(f"응답: {response.content[:200]}..." if len(response.content) > 200 else f"응답: {response.content}")
    except Exception as e:
        print(f"\n[{model_name}] 오류: {e}")

In [ ]:
# Anthropic Claude 모델 사용 (ANTHROPIC_API_KEY 필요)
if os.getenv("ANTHROPIC_API_KEY"):
    claude_model = init_chat_model(
        "anthropic:claude-sonnet-4-5-20250929",
        temperature=0
    )
    
    response = claude_model.invoke("AI 에이전트를 한 문장으로 설명해주세요.")
    print(f"[Claude 응답]\n{response.content}")
else:
    print("ANTHROPIC_API_KEY가 설정되지 않았습니다.")

## 5. 직접 모델 클래스 사용하기

`init_chat_model` 대신 직접 모델 클래스를 사용할 수도 있습니다. 이 방법은 더 세밀한 설정이 필요할 때 유용합니다.

In [ ]:
from langchain_openai import ChatOpenAI

# 직접 클래스 사용
model_direct = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=1000,
    timeout=30,
    max_retries=3,
    # 추가 파라미터
    model_kwargs={
        "top_p": 0.9,          # 누적 확률 샘플링
        "frequency_penalty": 0.5,  # 반복 억제
    }
)

response = model_direct.invoke("창의적인 AI 에이전트 이름 3개를 제안해주세요.")
print(response.content)

## 6. 메시지 형식 사용하기

Chat 모델은 다양한 메시지 형식을 지원합니다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = init_chat_model("gpt-4o-mini", temperature=0)

# 메시지 리스트로 호출
messages = [
    SystemMessage(content="당신은 친절한 한국어 튜터입니다."),
    HumanMessage(content="'인공지능'을 영어로 어떻게 말하나요?")
]

response = model.invoke(messages)
print(f"응답: {response.content}")

In [ ]:
# 대화 히스토리 포함
conversation = [
    SystemMessage(content="당신은 Python 프로그래밍 튜터입니다. 간결하게 답변하세요."),
    HumanMessage(content="리스트와 튜플의 차이점은?"),
    AIMessage(content="리스트는 변경 가능(mutable)하고, 튜플은 변경 불가능(immutable)합니다."),
    HumanMessage(content="예제 코드를 보여주세요.")
]

response = model.invoke(conversation)
print(response.content)

## 7. 비동기 호출

비동기 환경에서는 `ainvoke`를 사용합니다.

In [ ]:
import asyncio

model = init_chat_model("gpt-4o-mini", temperature=0)

async def ask_question(question: str) -> str:
    response = await model.ainvoke(question)
    return response.content

# 여러 질문을 동시에 처리
async def main():
    questions = [
        "Python의 장점은?",
        "JavaScript의 장점은?",
        "Rust의 장점은?"
    ]
    
    # 동시 실행
    results = await asyncio.gather(*[ask_question(q) for q in questions])
    
    for q, r in zip(questions, results):
        print(f"Q: {q}")
        print(f"A: {r[:100]}...\n")

# Jupyter에서 실행
await main()

## 8. 스트리밍 (Streaming)

스트리밍은 모델의 응답을 생성되는 대로 실시간으로 받아올 수 있게 해줍니다. `stream()` 메서드는 `AIMessageChunk` 객체를 순차적으로 반환하며, 이들을 합산(`+`)하여 전체 메시지를 구성할 수 있습니다.

In [ ]:
# 기본 텍스트 스트리밍
model = init_chat_model("gpt-4o-mini", temperature=0.7)

print("=== 스트리밍 응답 ===")
for chunk in model.stream("파이썬으로 간단한 웹 스크래퍼를 만드는 방법을 3단계로 설명해주세요."):
    print(chunk.content, end="", flush=True)

print("\n\n" + "=" * 60)

# 스트림 청크를 모아서 전체 메시지 구성하기
print("\n=== 청크 수집 ===")
full = None
for chunk in model.stream("AI 에이전트의 3가지 핵심 구성요소는?"):
    full = chunk if full is None else full + chunk

print(f"최종 메시지 길이: {len(full.content)} 문자")
print(f"최종 메시지:\n{full.content}")

## 9. 배치 처리 (Batch)

여러 독립적인 요청을 `batch()`로 일괄 처리하면 병렬 처리를 통해 성능을 개선할 수 있습니다.
`batch_as_completed()`를 사용하면 완료되는 대로 결과를 받을 수도 있습니다.

In [ ]:
model = init_chat_model("gpt-4o-mini", temperature=0)

# 여러 질문을 배치로 처리
questions = [
    "Python의 주요 특징 3가지는?",
    "JavaScript의 주요 특징 3가지는?",
    "Rust의 주요 특징 3가지는?",
    "Go의 주요 특징 3가지는?",
]

print("=== batch() 처리 ===\n")
responses = model.batch(questions)

for q, r in zip(questions, responses):
    print(f"질문: {q}")
    print(f"답변: {r.content[:100]}...")
    print("-" * 60)

In [ ]:
# batch_as_completed: 완료되는 대로 결과 받기
print("=== batch_as_completed() 처리 ===\n")

questions = [
    "간단한 Hello World를 한국어로 번역하면?",
    "머신러닝과 딥러닝의 차이를 한 문장으로?",
    "LLM이란?"
]

for idx, result in model.batch_as_completed(questions):
    print(f"[완료 - 인덱스 {idx}] {questions[idx]}")
    print(f"답변: {result.content[:80]}...\n")

## 10. Structured Output 맛보기

모델의 응답을 Pydantic 모델 등의 정해진 스키마에 맞춰 구조화된 형태로 받을 수 있습니다.
`with_structured_output()` 메서드를 사용합니다.

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """영화 정보"""
    title: str = Field(..., description="영화 제목")
    year: int = Field(..., description="개봉 연도")
    director: str = Field(..., description="감독 이름")
    rating: float = Field(..., description="평점 (0-10)")

model = init_chat_model("gpt-4o-mini")
model_with_structure = model.with_structured_output(Movie)

# 구조화된 응답 받기
response = model_with_structure.invoke("영화 '인셉션'에 대한 정보를 알려주세요.")
print(f"타입: {type(response)}")
print(f"\n제목: {response.title}")
print(f"연도: {response.year}")
print(f"감독: {response.director}")
print(f"평점: {response.rating}")

## 11. 멀티모달 입력 및 추론 영역 출력

최신 LLM 모델들은 텍스트뿐만 아니라 이미지 등의 멀티모달 입력을 처리하고, 추론 과정(thinking/reasoning)을 출력할 수 있습니다.

### 11.1 멀티모달 입력 (이미지)

GPT-4o, GPT-4o-mini, Claude Sonnet 등의 모델은 이미지를 입력으로 받을 수 있습니다.

In [ ]:
from langchain_core.messages import HumanMessage
import base64
import requests

# 예제 이미지 URL (공개 이미지)
image_url = "https://picsum.photos/200/300"

# 방법 1: 이미지 URL 직접 전달
model = init_chat_model("gpt-4o-mini")

message = HumanMessage(
    content=[
        {"type": "text", "text": "이 이미지를 설명해주세요."},
        {
            "type": "image_url",
            "image_url": {"url": image_url}
        }
    ]
)

response = model.invoke([message])
print(f"[URL 방식]\n{response.content}\n")

# 방법 2: Base64 인코딩된 이미지 전달
response = requests.get(image_url)
image_data = base64.b64encode(response.content).decode("utf-8")

message_base64 = HumanMessage(
    content=[
        {"type": "text", "text": "이 이미지의 주요 특징을 3가지만 말해주세요."},
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}
        }
    ]
)

response = model.invoke([message_base64])
print(f"[Base64 방식]\n{response.content}")

### 11.2 추론 영역 출력 (Thinking/Reasoning)

일부 모델들은 추론 과정을 출력할 수 있습니다:
- **Claude Sonnet 4.5 이상**: `extended_thinking=True` 옵션으로 추론 과정 출력
- **OpenAI o1/o3 시리즈**: 기본적으로 reasoning 과정 포함

추론 영역은 모델이 답변을 생성하기 전에 내부적으로 생각하는 과정을 보여줍니다.

### OpenAI 추론 모델

In [ ]:
# OpenAI gpt-5-mini Thinking 사용
from langchain_openai import ChatOpenAI

reasoning = {
    "effort": "medium",  # 'low', 'medium', or 'high'
    "summary": "auto",  # 'detailed', 'auto', or None
}
# extended_thinking 옵션 활성화
openai_thinking = init_chat_model(
    model="openai:gpt-5", reasoning=reasoning, output_version="responses/v1"
)

# 복잡한 문제를 던져서 추론 과정 확인
question = """
다음 수학 문제를 단계별로 풀어주세요:

한 농부가 닭과 소를 키우고 있습니다.
머리 수의 합은 20개이고, 다리 수의 합은 56개입니다.
닭과 소는 각각 몇 마리일까요?
"""
    
response = openai_thinking.invoke("question")

# Response text
print(f"답변: \n{response.text}")
print("-"*100)
# Reasoning summaries
print(f"추론:\n")
for block in response.content:
    if block["type"] == "reasoning":
        for summary in block["summary"]:
            print(summary["text"])

In [ ]:
print(response)

### Anthropic 추론 모델

In [ ]:
# Claude Sonnet 4.5 - Extended Thinking 사용
if os.getenv("ANTHROPIC_API_KEY"):
    # extended_thinking 옵션 활성화
    claude_thinking = init_chat_model(
        "anthropic:claude-sonnet-4-5-20250929",
        temperature=1.0,
        model_kwargs={
            "thinking": {"type": "enabled", "budget_tokens": 10_000}  # 추론 과정 출력 활성화
        }
    )
    
    # 복잡한 문제를 던져서 추론 과정 확인
    question = """
    다음 수학 문제를 단계별로 풀어주세요:
    
    한 농부가 닭과 소를 키우고 있습니다.
    머리 수의 합은 20개이고, 다리 수의 합은 56개입니다.
    닭과 소는 각각 몇 마리일까요?
    """
    
    response = claude_thinking.invoke(question)
    
    # thinking 블록과 text 블록 분리 출력
    if isinstance(response.content, list):
        for i, block in enumerate(response.content):
            if isinstance(block, dict):
                if block.get('type') == 'thinking':
                    print(f"[추론 과정]")
                    print("-" * 60)
                    print(block.get('thinking', '')[:500])
                    print("...\n")
                elif block.get('type') == 'text':
                    print(f"[최종 답변]")
                    print("-" * 60)
                    print(block.get('text', ''))
    else:
        print(response.content)
else:
    print("ANTHROPIC_API_KEY가 설정되지 않았습니다.")

### 11.3 추론 과정 활용하기

추론 과정은 다음과 같은 상황에서 유용합니다:
- 복잡한 수학 문제 해결
- 논리적 추론이 필요한 문제
- 단계별 분석이 중요한 작업
- 디버깅 및 문제 해결 과정 이해

In [ ]:
# 추론 과정을 활용한 실전 예제
if os.getenv("ANTHROPIC_API_KEY"):
    model_with_thinking = init_chat_model(
        "anthropic:claude-sonnet-4-5-20250929",
        temperature=1.0,
        model_kwargs={
            "thinking": {"type": "enabled", "budget_tokens": 3000}  # 추론 과정 출력 활성화
                      }
    )
    
    # 복잡한 코드 디버깅 시나리오
    code_with_bug = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)

# 테스트
print(calculate_average([]))  # 문제 발생!
"""
    
    question = f"""
다음 Python 코드에 버그가 있습니다. 
문제점을 찾고, 왜 문제인지 설명하고, 수정 방법을 제시해주세요.

```python
{code_with_bug}
```
"""
    
    response = model_with_thinking.invoke(question)
    
    print("=== 코드 디버깅 with Extended Thinking ===\n")
    
    # thinking 블록과 text 블록 분리 출력
    if isinstance(response.content, list):
        for block in response.content:
            if isinstance(block, dict):
                if block.get('type') == 'thinking':
                    print("[모델의 내부 추론 과정]")
                    print("-" * 60)
                    print(block.get('thinking', '')[:800])
                    print("...\n")
                elif block.get('type') == 'text':
                    print("[최종 답변]")
                    print("-" * 60)
                    print(block.get('text', ''))
    else:
        print(response.content)
else:
    print("ANTHROPIC_API_KEY가 설정되어 있지 않아 extended thinking 예제를 실행할 수 없습니다.")

## 12. 요약

### 이 노트북에서 배운 것

1. **init_chat_model 기본 사용법**
   - `init_chat_model("model_name")` 또는 `init_chat_model("provider:model_name")`
   - 최신 모델: GPT-4o, GPT-4o-mini, Claude Sonnet 4.5, Gemini 2.5 등
   
2. **주요 파라미터**
   - `temperature`: 창의성 조절 (0~2)
   - `max_tokens`: 최대 출력 길이
   - `timeout`: 타임아웃 설정
   - `max_retries`: 재시도 횟수

3. **호출 방식 (Invocation Methods)**
   - **invoke**: 단일 요청으로 전체 응답 받기
   - **stream**: 응답을 실시간으로 스트리밍 (AIMessageChunk)
   - **batch**: 여러 요청을 배치로 병렬 처리
   - **batch_as_completed**: 완료되는 대로 결과 받기
   - **ainvoke**: 비동기 호출

4. **메시지 형식**
   - 문자열로 직접 호출
   - 메시지 리스트로 호출 (SystemMessage, HumanMessage, AIMessage)
   - 대화 히스토리 관리

5. **직접 모델 클래스 사용**
   - `ChatOpenAI`, `ChatAnthropic` 등으로 더 세밀한 설정 가능
   - `model_kwargs`로 제공자별 고급 옵션 설정

6. **Structured Output**
   - Pydantic 모델로 구조화된 응답 받기
   - `with_structured_output()` 메서드 활용

7. **멀티모달 입력**
   - 이미지 URL 또는 Base64 인코딩으로 이미지 전달
   - content 리스트에 text와 image_url 블록 혼합

8. **추론 영역 출력 (Thinking/Reasoning)**
   - Claude: `extended_thinking=True`로 추론 과정 확인
   - OpenAI o1/o3: 기본적으로 reasoning 포함
   - content에서 thinking 타입 블록으로 접근

### 참고 자료
- [LangChain Models 공식 문서](https://docs.langchain.com/oss/python/langchain/models)
- [LangChain Chat Model Integrations](https://docs.langchain.com/oss/python/integrations/chat)

### 다음 단계
- 부록_03: Tool 데코레이터와 ToolRuntime 활용하기